In [1]:
from pathlib import Path
import h5py
import pandas as pd
from IPython.display import display

# --------------------------------------------------
# AYAR
# --------------------------------------------------
work_dir = Path(r"C:\Users\enesz\Desktop\mintpy\Bozuyuk_AOI_5km")

target_files = [
    "velocity.h5",
    "temporalCoherence.h5",
    "timeseries.h5",
    "avgSpatialCoh.h5",
    "avgPhaseVelocity.h5",
    "numTriNonzeroIntAmbiguity.h5",
    "numInvIfgram.h5",
    "timeseries_demErr.h5",
    "inputs/ifgramStack.h5",
    "inputs/geometryGeo.h5",
]

# --------------------------------------------------
# YARDIMCI FONKSİYONLAR
# --------------------------------------------------
def to_text(v):
    if isinstance(v, bytes):
        return v.decode("utf-8", errors="ignore")
    return str(v)

def inspect_h5(path: Path):
    result = {
        "file": path.name,
        "exists": path.exists(),
        "file_type": None,
        "length": None,
        "width": None,
        "unit": None,
        "dataset_count": 0,
        "has_velocityStd": False,
        "datasets": []
    }

    if not path.exists():
        return result

    with h5py.File(path, "r") as f:
        # Kök attribute'lar
        attrs = {k: to_text(v) for k, v in f.attrs.items()}
        result["file_type"] = attrs.get("FILE_TYPE")
        result["length"] = attrs.get("LENGTH")
        result["width"] = attrs.get("WIDTH")
        result["unit"] = attrs.get("UNIT")

        # Tüm dataset'leri tara
        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                item = {
                    "name": name,
                    "shape": tuple(obj.shape),
                    "dtype": str(obj.dtype)
                }
                result["datasets"].append(item)

        f.visititems(visitor)

    result["dataset_count"] = len(result["datasets"])
    result["has_velocityStd"] = any("velocityStd" in d["name"] for d in result["datasets"])
    return result

# --------------------------------------------------
# DOSYALARI İNCELE
# --------------------------------------------------
summaries = []
all_dataset_rows = []

for rel in target_files:
    full_path = work_dir / rel
    info = inspect_h5(full_path)
    summaries.append({
        "file": rel,
        "exists": info["exists"],
        "FILE_TYPE": info["file_type"],
        "LENGTH": info["length"],
        "WIDTH": info["width"],
        "UNIT": info["unit"],
        "dataset_count": info["dataset_count"],
        "has_velocityStd": info["has_velocityStd"],
    })

    for d in info["datasets"]:
        all_dataset_rows.append({
            "file": rel,
            "dataset": d["name"],
            "shape": str(d["shape"]),
            "dtype": d["dtype"],
        })

summary_df = pd.DataFrame(summaries)
datasets_df = pd.DataFrame(all_dataset_rows)

print("Çalışma klasörü:", work_dir)
print("\nDosya özeti:")
display(summary_df)

print("\nTüm dataset listesi:")
display(datasets_df)

# --------------------------------------------------
# ÖZEL KONTROL: ANA DOSYALAR
# --------------------------------------------------
print("\n--- Hızlı kontrol ---")

for important in ["velocity.h5", "temporalCoherence.h5", "timeseries.h5"]:
    sub = datasets_df[datasets_df["file"] == important]
    print(f"\n{important}")
    if sub.empty:
        print("  Dosya yok veya dataset bulunamadı.")
    else:
        for _, row in sub.iterrows():
            print(f"  - {row['dataset']} | shape={row['shape']} | dtype={row['dtype']}")

Çalışma klasörü: C:\Users\enesz\Desktop\mintpy\Bozuyuk_AOI_5km

Dosya özeti:


,file,exists,FILE_TYPE,LENGTH,WIDTH,UNIT,dataset_count,has_velocityStd
0,velocity.h5,True,velocity,501,501,m/year,5,True
1,temporalCoherence.h5,True,temporalCoherence,501,501,1,1,False
2,timeseries.h5,True,timeseries,501,501,m,3,False
3,avgSpatialCoh.h5,True,coherence,501,501,1,1,False
4,avgPhaseVelocity.h5,True,velocity,501,501,m/year,1,False
5,numTriNonzeroIntAmbiguity.h5,True,mask,501,501,1,1,False
6,numInvIfgram.h5,True,mask,501,501,1,1,False
7,timeseries_demErr.h5,True,timeseries,501,501,m,3,False
8,inputs/ifgramStack.h5,True,ifgramStack,501,501,None,5,False
9,inputs/geometryGeo.h5,True,geometry,501,501,None,4,False



Tüm dataset listesi:


,file,dataset,shape,dtype
0,velocity.h5,intercept,"(501, 501)",float32
1,velocity.h5,interceptStd,"(501, 501)",float32
2,velocity.h5,residue,"(501, 501)",float32
3,velocity.h5,velocity,"(501, 501)",float32
4,velocity.h5,velocityStd,"(501, 501)",float32
5,temporalCoherence.h5,temporalCoherence,"(501, 501)",float32
6,timeseries.h5,bperp,"(119,)",float32
7,timeseries.h5,date,"(119,)",|S8
8,timeseries.h5,timeseries,"(119, 501, 501)",float32
9,avgSpatialCoh.h5,coherence,"(501, 501)",float32



--- Hızlı kontrol ---

velocity.h5
  - intercept | shape=(501, 501) | dtype=float32
  - interceptStd | shape=(501, 501) | dtype=float32
  - residue | shape=(501, 501) | dtype=float32
  - velocity | shape=(501, 501) | dtype=float32
  - velocityStd | shape=(501, 501) | dtype=float32

temporalCoherence.h5
  - temporalCoherence | shape=(501, 501) | dtype=float32

timeseries.h5
  - bperp | shape=(119,) | dtype=float32
  - date | shape=(119,) | dtype=|S8
  - timeseries | shape=(119, 501, 501) | dtype=float32


In [6]:
from pathlib import Path

project_root = Path(r"C:\Users\enesz\Desktop\insar-dashboard")
project_root.mkdir(parents=True, exist_ok=True)

# 1) requirements.txt içeriği
requirements_text = """streamlit>=1.42
numpy>=1.26
plotly>=5.0
h5py>=3.10
"""

# 2) En küçük çalışan app.py
app_code = """import streamlit as st

st.set_page_config(page_title="Bozüyük InSAR Tabanlı Deformasyon Takip Arayüzü", layout="wide")
st.title("Bozuyuk InSAR Dashboard")
st.success("Uygulama çalışıyor.")
"""

# Dosyaları yaz
(project_root / "requirements.txt").write_text(requirements_text, encoding="utf-8")
(project_root / "app.py").write_text(app_code, encoding="utf-8")

print("Yazıldı:")
print(project_root / "requirements.txt")
print(project_root / "app.py")
print("\\nSonra terminalde şu komutu çalıştır:")
print(f'streamlit run "{project_root / "app.py"}"')

Yazıldı:
C:\Users\enesz\Desktop\insar-dashboard\requirements.txt
C:\Users\enesz\Desktop\insar-dashboard\app.py
\nSonra terminalde şu komutu çalıştır:
streamlit run "C:\Users\enesz\Desktop\insar-dashboard\app.py"


In [14]:
from pathlib import Path

project_root = Path(r"C:\Users\enesz\Desktop\insar-dashboard")
app_path = project_root / "app.py"

app_code = r'''import streamlit as st
from pathlib import Path
import numpy as np
import h5py
import pandas as pd
import folium
from folium.raster_layers import ImageOverlay
from folium.features import DivIcon
from streamlit_folium import st_folium
from pyproj import Transformer
from matplotlib import cm, colors

st.set_page_config(page_title="Bozuyuk InSAR Dashboard", layout="wide")

st.title("Bozuyuk InSAR Dashboard")
st.caption("True-map overlay for MintPy velocity / coherence")

# --------------------------------------------------
# AOI
# --------------------------------------------------
AOI_CENTER_LAT = 39.86891807427488
AOI_CENTER_LON = 30.17805156594522
AOI_RADIUS_M = 5000

# --------------------------------------------------
# DATA PATH
# --------------------------------------------------
mintpy_dir = Path(r"C:\Users\enesz\Desktop\mintpy\Bozuyuk_AOI_5km")
velocity_h5 = mintpy_dir / "velocity.h5"
coh_h5 = mintpy_dir / "temporalCoherence.h5"
ts_h5 = mintpy_dir / "timeseries.h5"

# --------------------------------------------------
# HELPERS
# --------------------------------------------------
def _to_text(v):
    if isinstance(v, bytes):
        return v.decode("utf-8", errors="ignore")
    return str(v)

@st.cache_data
def load_h5_dataset(file_path: str, dataset_name: str):
    with h5py.File(file_path, "r") as f:
        arr = f[dataset_name][:].astype("float32")
        attrs = {k: _to_text(v) for k, v in f.attrs.items()}
    arr = np.where(np.isfinite(arr), arr, np.nan)
    return arr, attrs

@st.cache_data
def load_dates(file_path: str):
    with h5py.File(file_path, "r") as f:
        raw_dates = f["date"][:]
    dates = pd.to_datetime([d.decode("utf-8") for d in raw_dates], format="%Y%m%d")
    return dates

def get_projected_axes(attrs, shape):
    x_first = float(attrs["X_FIRST"])
    y_first = float(attrs["Y_FIRST"])
    x_step = float(attrs["X_STEP"])
    y_step = float(attrs["Y_STEP"])
    h, w = shape
    xs = x_first + np.arange(w) * x_step
    ys = y_first + np.arange(h) * y_step
    return xs, ys, x_first, y_first, x_step, y_step

def get_projected_bounds(attrs, shape):
    xs, ys, x_first, y_first, x_step, y_step = get_projected_axes(attrs, shape)
    h, w = shape
    x_last = x_first + (w - 1) * x_step
    y_last = y_first + (h - 1) * y_step

    west  = min(x_first, x_last) - abs(x_step) / 2.0
    east  = max(x_first, x_last) + abs(x_step) / 2.0
    south = min(y_first, y_last) - abs(y_step) / 2.0
    north = max(y_first, y_last) + abs(y_step) / 2.0
    return west, south, east, north

def projected_bounds_to_latlon(bounds_proj, src_epsg):
    west, south, east, north = bounds_proj
    t = Transformer.from_crs(src_epsg, "EPSG:4326", always_xy=True)

    sw_lon, sw_lat = t.transform(west, south)
    ne_lon, ne_lat = t.transform(east, north)

    return [[sw_lat, sw_lon], [ne_lat, ne_lon]]

def array_to_rgba(arr, cmap_name, vmin, vmax):
    arr2 = np.array(arr, dtype="float32")
    valid = np.isfinite(arr2)

    norm = colors.Normalize(vmin=vmin, vmax=vmax, clip=True)
    cmap = cm.get_cmap(cmap_name)

    rgba = cmap(norm(np.nan_to_num(arr2, nan=vmin)))
    rgba[..., 3] = np.where(valid, 1.0, 0.0)   # nan alanları şeffaf
    rgba = (rgba * 255).astype("uint8")
    return rgba

def nearest_pixel_from_latlon(lat, lon, xs_proj, ys_proj, src_epsg):
    t = Transformer.from_crs("EPSG:4326", src_epsg, always_xy=True)
    x_proj, y_proj = t.transform(lon, lat)

    col = int(np.argmin(np.abs(xs_proj - x_proj)))
    row = int(np.argmin(np.abs(ys_proj - y_proj)))
    return row, col, x_proj, y_proj

# --------------------------------------------------
# FILE CHECK
# --------------------------------------------------
if not velocity_h5.exists() or not coh_h5.exists() or not ts_h5.exists():
    st.error("Gerekli MintPy dosyaları bulunamadı.")
    st.stop()

# --------------------------------------------------
# LOAD
# --------------------------------------------------
velocity_m_yr, vel_attrs = load_h5_dataset(str(velocity_h5), "velocity")
velocity_std_m_yr, _ = load_h5_dataset(str(velocity_h5), "velocityStd")
temporal_coh, coh_attrs = load_h5_dataset(str(coh_h5), "temporalCoherence")
dates = load_dates(str(ts_h5))

velocity_mm_yr = velocity_m_yr * 1000.0
velocity_std_mm_yr = velocity_std_m_yr * 1000.0

# --------------------------------------------------
# SIDEBAR
# --------------------------------------------------
st.sidebar.header("Coverage")
st.sidebar.write(f"Dates: **{dates.min().date()} → {dates.max().date()}**")
st.sidebar.write(f"Epoch count: **{len(dates)}**")

st.sidebar.header("Display")
overlay_name = st.sidebar.radio("Overlay", ["Velocity", "Temporal Coherence"], index=0)
opacity = st.sidebar.slider("Overlay opacity", 0.0, 1.0, 0.65, 0.05)

# Bu AOI için değerler UTM 35N ile uyumlu görünüyor.
src_epsg = st.sidebar.selectbox(
    "Projected CRS",
    ["EPSG:32635", "EPSG:23035"],
    index=0
)

coh_thr = st.sidebar.slider("Min temporal coherence", 0.00, 1.00, 0.70, 0.01)
std_thr = st.sidebar.number_input("Max velocityStd (mm/y)", min_value=0.0, value=10.0, step=0.5)

vmin = st.sidebar.number_input("Velocity min (mm/y)", value=-60.0, step=5.0)
vmax = st.sidebar.number_input("Velocity max (mm/y)", value=60.0, step=5.0)

# --------------------------------------------------
# FILTER
# --------------------------------------------------
mask = (
    np.isfinite(velocity_mm_yr)
    & np.isfinite(velocity_std_mm_yr)
    & np.isfinite(temporal_coh)
    & (temporal_coh >= coh_thr)
    & (velocity_std_mm_yr <= std_thr)
)

velocity_filtered = np.where(mask, velocity_mm_yr, np.nan)

# --------------------------------------------------
# METRICS
# --------------------------------------------------
valid_vel = velocity_filtered[np.isfinite(velocity_filtered)]
valid_coh = temporal_coh[np.isfinite(temporal_coh)]

m1, m2, m3, m4 = st.columns(4)
m1.metric("Min velocity (mm/y)", f"{np.nanmin(valid_vel):.2f}" if valid_vel.size else "NA")
m2.metric("Max velocity (mm/y)", f"{np.nanmax(valid_vel):.2f}" if valid_vel.size else "NA")
m3.metric("Mean coherence", f"{np.nanmean(valid_coh):.3f}" if valid_coh.size else "NA")
m4.metric("Valid filtered pixels", f"{np.isfinite(velocity_filtered).sum():,}")

# --------------------------------------------------
# SPATIAL GEOMETRY
# --------------------------------------------------
xs_proj, ys_proj, *_ = get_projected_axes(vel_attrs, velocity_filtered.shape)
bounds_proj = get_projected_bounds(vel_attrs, velocity_filtered.shape)
bounds_latlon = projected_bounds_to_latlon(bounds_proj, src_epsg)

# --------------------------------------------------
# OVERLAY IMAGE
# --------------------------------------------------
if overlay_name == "Velocity":
    overlay_arr = velocity_filtered
    overlay_rgba = array_to_rgba(overlay_arr, "RdBu_r", vmin, vmax)
    overlay_caption = f"Velocity overlay ({vmin} to {vmax} mm/y)"
else:
    overlay_arr = temporal_coh
    overlay_rgba = array_to_rgba(overlay_arr, "viridis", 0.0, 1.0)
    overlay_caption = "Temporal coherence overlay (0 to 1)"

st.write(f"**{overlay_caption}**")

# --------------------------------------------------
# TRUE MAP
# --------------------------------------------------
m = folium.Map(
    location=[AOI_CENTER_LAT, AOI_CENTER_LON],
    zoom_start=12,
    tiles="CartoDB positron",
    control_scale=True
)

folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Satellite",
    overlay=False,
    control=True
).add_to(m)

ImageOverlay(
    image=overlay_rgba,
    bounds=bounds_latlon,
    opacity=opacity,
    interactive=True,
    cross_origin=False,
    name=overlay_name
).add_to(m)

# AOI center star
folium.Marker(
    location=[AOI_CENTER_LAT, AOI_CENTER_LON],
    icon=DivIcon(
        icon_size=(24, 24),
        icon_anchor=(12, 12),
        html='<div style="font-size:22px;color:gold;text-shadow:0 0 3px black;">★</div>'
    ),
    tooltip="AOI center"
).add_to(m)

# AOI radius
folium.Circle(
    location=[AOI_CENTER_LAT, AOI_CENTER_LON],
    radius=AOI_RADIUS_M,
    color="gold",
    weight=2,
    fill=False,
    tooltip="AOI radius = 5 km"
).add_to(m)

folium.LayerControl().add_to(m)

map_state = st_folium(
    m,
    height=720,
    width=None,
    returned_objects=["last_clicked", "zoom", "bounds"]
)

# --------------------------------------------------
# CLICK INFO
# --------------------------------------------------
clicked = map_state.get("last_clicked", None)

if clicked:
    click_lat = clicked["lat"]
    click_lon = clicked["lng"]

    row, col, x_proj, y_proj = nearest_pixel_from_latlon(
        click_lat, click_lon, xs_proj, ys_proj, src_epsg
    )

    row = max(0, min(row, velocity_filtered.shape[0] - 1))
    col = max(0, min(col, velocity_filtered.shape[1] - 1))

    vel_val = velocity_filtered[row, col]
    coh_val = temporal_coh[row, col]
    std_val = velocity_std_mm_yr[row, col]

    st.subheader("Clicked location")
    c1, c2, c3 = st.columns(3)
    c1.write(f"Lat / Lon: **{click_lat:.6f}, {click_lon:.6f}**")
    c2.write(f"Projected X / Y: **{x_proj:.1f}, {y_proj:.1f}**")
    c3.write(f"Row / Col: **{row}, {col}**")

    d1, d2, d3 = st.columns(3)
    d1.metric("Velocity (mm/y)", f"{vel_val:.3f}" if np.isfinite(vel_val) else "NA")
    d2.metric("Temporal coherence", f"{coh_val:.3f}" if np.isfinite(coh_val) else "NA")
    d3.metric("VelocityStd (mm/y)", f"{std_val:.3f}" if np.isfinite(std_val) else "NA")
else:
    st.info("Harita üzerinde bir noktaya tıkla. Burada gerçek lat/lon ve piksel değerleri görünecek.")
'''

app_path.write_text(app_code, encoding="utf-8")

print("Güncellendi:", app_path)
print("\nGerekirse şu paketleri kur:")
print("pip install folium streamlit-folium pyproj matplotlib")
print("\nSonra çalıştır:")
print(f'streamlit run "{app_path}"')

Güncellendi: C:\Users\enesz\Desktop\insar-dashboard\app.py

Gerekirse şu paketleri kur:
pip install folium streamlit-folium pyproj matplotlib

Sonra çalıştır:
streamlit run "C:\Users\enesz\Desktop\insar-dashboard\app.py"


In [16]:
streamlit>=1.42
numpy>=1.26
plotly>=5.0
h5py>=3.10
folium>=0.16
streamlit-folium>=0.20
pyproj>=3.6
matplotlib>=3.8

NameError: name 'streamlit' is not defined